In [1]:
# import torch
import datajoint as dj
from experiments.dj.dataloader_tables import DataLoaderConfig, AltDataLoaderConfig
from experiments.dj.schema import schema
from experiments.dj.likelihood_tables import LikelihoodConfig

from experiments.dj.prior_tables import FlowPriorConfig, AdaptPriorConfig
from experiments.dj.sysident_tables import SIConfig
from experiments.dj.posterior_tables import SBVGPConfig
from experiments.dj.result_tables import (
    FlowPriorResult,
    FPSamples,
    FPSamplesConfig,
    LikelihoodResult,
    MLPCondSamples,
    MLPCondSamplesConfig,
    SBVGPResult2,
    SIResult,
    AdaptPriorResult,
    AdaptPriorSamplesConfig,
    SBVGPAdaptedResult,
)
from experiments.dj.evaluation_tables import SIEval, SBVGPEval
from experiments.dj.trainer_tables import (
    FPTrainerConfig,
    LLTrainerConfig,
    SBVGPTrainerConfig,
    SITrainerConfig,
    AdaptPriorTrainer,
)
from experiments.dj.dj_helpers import drop_schema_dot_jobs
from experiments.dj.dj_helpers import fetch_best_model_results
from task_transfer.ml_lib.data_loading import build_dataloaders

from collections import OrderedDict

from task_transfer.utils.utils import dict_product
from experiments.dj.evaluation_tables import FlowPriorEval

[2024-09-05 10:15:32,291][INFO]: Connecting sshrinivasan@134.76.19.44:3306
[2024-09-05 10:15:32,299][INFO]: Connected sshrinivasan@134.76.19.44:3306


In [6]:
SIResult 

si_id,trainer_id,dl_id,"train_ll_mean mean per dimension, per sample, in nats",train_ll_sem standard error of the mean,val_ll_mean,val_ll_sem,test_ll_mean,test_ll_sem,tracker_output,eval_output,model
39f03eac90b439e7897b6f4a65464417,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a,-41.9784049987793,1.2008603811264038,-42.252342224121094,2.2518041133880615,-42.61870193481445,3.1429383754730225,=BLOB=,=BLOB=,=BLOB=
45f442bf0f2c1dcfb984300f172610bf,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a,-42.232418060302734,1.1933763027191162,-42.48735809326172,2.243901014328003,-42.891815185546875,3.142155170440674,=BLOB=,=BLOB=,=BLOB=
4babfd012e0443ef1ae2139deb4def7b,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a,-42.910369873046875,1.1807067394256592,-43.04535675048828,2.2083466053009033,-43.36171340942383,3.089459180831909,=BLOB=,=BLOB=,=BLOB=
5531fc5da506b401bcdb7f43e282f731,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a,-42.95983123779297,1.1811304092407227,-43.03318786621094,2.2080531120300293,-43.36014938354492,3.083613157272339,=BLOB=,=BLOB=,=BLOB=
5c533709d4d10462567539f3529e43b4,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a,-42.945709228515625,1.1790655851364136,-43.054447174072266,2.2038252353668213,-43.395957946777344,3.0836145877838135,=BLOB=,=BLOB=,=BLOB=
6deb07682dd4919b2ebe01c255b23445,a2a0e79d4c785fead2a84d56420b6063,5352c4a57ef18797b082283de593157b,-2.799105167388916,0.312860369682312,-2.7971572875976562,0.5990963578224182,-2.8339250087738037,0.9535223245620728,=BLOB=,=BLOB=,=BLOB=
6deb07682dd4919b2ebe01c255b23445,a2a0e79d4c785fead2a84d56420b6063,8e9be142eedb21007255e89dbff362da,-0.5238828659057617,0.3509235084056854,-0.5353844165802002,0.5468757748603821,-0.542957603931427,0.8958107233047485,=BLOB=,=BLOB=,=BLOB=
6deb07682dd4919b2ebe01c255b23445,a2a0e79d4c785fead2a84d56420b6063,94efb58694007205fac996d7963f88c5,-1.1356943845748901,0.18121908605098724,-1.1543511152267456,0.3394978642463684,-1.1744518280029297,0.48188719153404236,=BLOB=,=BLOB=,=BLOB=
6deb07682dd4919b2ebe01c255b23445,a2a0e79d4c785fead2a84d56420b6063,bb9bdd1ccd59e5a8c801d7f2d43e0317,-1.544669270515442,0.3287414610385895,-1.5727845430374146,0.6314698457717896,-1.6489523649215698,0.8309184908866882,=BLOB=,=BLOB=,=BLOB=
6deb07682dd4919b2ebe01c255b23445,a2a0e79d4c785fead2a84d56420b6063,d74090584b0b974c4444a5ec64c3d87d,-0.8725296258926392,0.24050629138946533,-0.9208231568336487,0.4479599893093109,-0.8024278283119202,0.6012524962425232,=BLOB=,=BLOB=,=BLOB=


In [6]:
DataLoaderConfig & "id = '260a5ea8175f75eaef132f42873ad14a'"

id,data_fname,train_prop,val_prop
260a5ea8175f75eaef132f42873ad14a,/src/project/data/synthetic/haefner_2afc/original_haefner_2afc_task_1_dataset.pkl,0.7,0.2


In [4]:
AdaptPriorResult & "dl_id = '9ef3ae6fea33eba634d928a88b866836'"

seed,prior_fp_id to index into FlowPriorConfig,prior_trainer_id to index into FPTrainerConfig,likelihood_id to index into LikelihoodConfig,likelihood_trainer_id to index into LLTrainerConfig,orig_dl_id to index into DataLoaderConfig used for the prior and likelihood training,trainer_id,dl_id,"train_marginal_obs_ll_mean mean per trial, per sample, in nats",train_marginal_obs_ll_sem standard error of the mean,val_marginal_obs_ll_mean,val_marginal_obs_ll_sem,test_marginal_obs_ll_mean,test_marginal_obs_ll_sem,"train_prior_ll_mean mean per trial, per sample, in nats",train_prior_ll_sem standard error of the mean,val_prior_ll_mean,val_prior_ll_sem,test_prior_ll_mean,test_prior_ll_sem,tracker_output,eval_output,model trained joint model NOT just the prior
-100,d0cf491f03b7f839c8a54834a6168081,c40a50ce9c77369770dddd5129836477,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a,e267b2071bca2c3f9431f155e8e58f23,9ef3ae6fea33eba634d928a88b866836,-5203.6923828125,593.451171875,-5200.5927734375,1111.11376953125,-5184.78564453125,1497.9891357421875,-225.7065887451172,3.2274301052093506,-225.50997924804688,6.009249210357666,-224.40745544433594,8.13746452331543,=BLOB=,=BLOB=,=BLOB=
42,d0cf491f03b7f839c8a54834a6168081,c40a50ce9c77369770dddd5129836477,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a,e267b2071bca2c3f9431f155e8e58f23,9ef3ae6fea33eba634d928a88b866836,-4989.09033203125,581.8089599609375,-4902.71240234375,996.241455078125,-4779.85595703125,1295.3453369140625,-123.60945892333984,2.029911994934082,-123.82306671142578,3.7798922061920166,-123.90569305419922,5.306024074554443,=BLOB=,=BLOB=,=BLOB=


In [10]:
FlowPriorResult & "dl_id = '260a5ea8175f75eaef132f42873ad14a'"

fp_id,trainer_id,dl_id,"train_ll_mean mean per dimension, per sample, in nats",train_ll_sem standard error of the mean,val_ll_mean,val_ll_sem,test_ll_mean,test_ll_sem,tracker_output,eval_output,model
0173a9aecec7a4e144011e0d3ff0c8a7,c40a50ce9c77369770dddd5129836477,260a5ea8175f75eaef132f42873ad14a,-51.72239303588867,1.2013747692108154,-51.840248107910156,2.282120943069458,-51.840248107910156,2.282120943069458,=BLOB=,=BLOB=,=BLOB=
0173a9aecec7a4e144011e0d3ff0c8a7,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a,-52.199527740478516,1.2041391134262085,-52.2991943359375,2.289771556854248,-52.2991943359375,2.289771556854248,=BLOB=,=BLOB=,=BLOB=
036827f08f256e836cacde3d2954ea4a,c40a50ce9c77369770dddd5129836477,260a5ea8175f75eaef132f42873ad14a,-72.07315826416016,1.3810926675796509,-71.84971618652344,2.6091067790985107,-71.84971618652344,2.6091067790985107,=BLOB=,=BLOB=,=BLOB=
036827f08f256e836cacde3d2954ea4a,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a,-72.36489868164062,1.463006615638733,-72.16575622558594,2.690886974334717,-72.16575622558594,2.690886974334717,=BLOB=,=BLOB=,=BLOB=
0626e82eec85ade7041703d85e168d5b,c40a50ce9c77369770dddd5129836477,260a5ea8175f75eaef132f42873ad14a,-126.7267837524414,1.5894485712051392,-126.70783996582031,2.9417123794555664,-126.70783996582031,2.9417123794555664,=BLOB=,=BLOB=,=BLOB=
0626e82eec85ade7041703d85e168d5b,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a,-123.26030731201172,1.62436842918396,-123.29173278808594,3.0070111751556396,-123.29173278808594,3.0070111751556396,=BLOB=,=BLOB=,=BLOB=
06bb4905529cab2d09a6b5fb479cd700,c40a50ce9c77369770dddd5129836477,260a5ea8175f75eaef132f42873ad14a,-74.34554290771484,1.4281342029571533,-74.0283432006836,2.6899912357330322,-74.0283432006836,2.6899912357330322,=BLOB=,=BLOB=,=BLOB=
06bb4905529cab2d09a6b5fb479cd700,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a,-71.88994598388672,1.4057899713516235,-71.6804428100586,2.6343274116516113,-71.6804428100586,2.6343274116516113,=BLOB=,=BLOB=,=BLOB=
06fe441dd0fdca3ae7bfa7a4ad856467,c40a50ce9c77369770dddd5129836477,260a5ea8175f75eaef132f42873ad14a,-55.91200256347656,1.221100091934204,-56.58893966674805,2.382314443588257,-56.58893966674805,2.382314443588257,=BLOB=,=BLOB=,=BLOB=
06fe441dd0fdca3ae7bfa7a4ad856467,f89651063b51487dcdf4041336ef89db,260a5ea8175f75eaef132f42873ad14a,-5981.26806640625,212.3249969482422,-6216.796875,713.691162109375,-6216.796875,713.691162109375,=BLOB=,=BLOB=,=BLOB=


In [ ]:
(AdaptPriorResult & "dl_id = '9ef3ae6fea33eba634d928a88b866836'")

In [2]:
AltDataLoaderConfig()

id,data_fname,train_prop,val_prop
260a5ea8175f75eaef132f42873ad14a,/src/project/data/synthetic/haefner_2afc/original_haefner_2afc_task_1_dataset.pkl,0.7,0.2
3d740ef65d4ec3d651cb862eb90143df,/src/project/data/synthetic/haefner_2afc/haefner_model_4neuron_task2_dataset.pkl,0.7,0.2
5352c4a57ef18797b082283de593157b,/src/project/data/synthetic/haefner_2afc/haefner_model_4neuron_highdelta_task2_dataset.pkl,0.7,0.2
592885da0624c8a8c3073ec47d9bcfba,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_task2_dataset.pkl,0.7,0.2
94efb58694007205fac996d7963f88c5,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_highdelta_task2_dataset.pkl,0.7,0.2
9ef3ae6fea33eba634d928a88b866836,/src/project/data/synthetic/haefner_2afc/original_haefner_2afc_task_2_dataset.pkl,0.7,0.2
b8379e7d6998fc94a08a9a3742eec12d,/src/project/data/synthetic/haefner_2afc/flat_haefner_dataset.pkl,0.7,0.2
d6b36dc9d4024882e4b7ccc597495a32,/src/project/data/synthetic/haefner_2afc/flat_haefner_100k_dataset.pkl,0.7,0.2
d74090584b0b974c4444a5ec64c3d87d,/src/project/data/synthetic/haefner_2afc/haefner_model_2neuron_highdelta_task2_dataset.pkl,0.7,0.2
e248f1c663817cdabf379015476a3a2e,/src/project/data/synthetic/haefner_2afc/haefner_model_2neuron_task2_dataset.pkl,0.7,0.2


In [2]:
DataLoaderConfig()

id,data_fname,train_prop,val_prop
05977a317062b759857ee411a2e60648,/src/project/data/synthetic/haefner_2afc/haefner_2neuron_task1.pkl,0.7,0.2
260a5ea8175f75eaef132f42873ad14a,/src/project/data/synthetic/haefner_2afc/original_haefner_2afc_task_1_dataset.pkl,0.7,0.2
4477b5e82704db0bc19727864c7ef5aa,/src/project/data/synthetic/haefner_2afc/haefner_4neuron_task1.pkl,0.7,0.2
5352c4a57ef18797b082283de593157b,/src/project/data/synthetic/haefner_2afc/haefner_model_4neuron_highdelta_task2_dataset.pkl,0.7,0.2
8e9be142eedb21007255e89dbff362da,/src/project/data/synthetic/haefner_2afc/haefner_model_2neuron_highdelta_task1_dataset.pkl,0.7,0.2
94efb58694007205fac996d7963f88c5,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_highdelta_task2_dataset.pkl,0.7,0.2
b8379e7d6998fc94a08a9a3742eec12d,/src/project/data/synthetic/haefner_2afc/flat_haefner_dataset.pkl,0.7,0.2
bb9bdd1ccd59e5a8c801d7f2d43e0317,/src/project/data/synthetic/haefner_2afc/haefner_model_4neuron_highdelta_task1_dataset.pkl,0.7,0.2
d74090584b0b974c4444a5ec64c3d87d,/src/project/data/synthetic/haefner_2afc/haefner_model_2neuron_highdelta_task2_dataset.pkl,0.7,0.2
f1ae78885d2ace1ba976199d4cf1a4d6,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_task1_dataset.pkl,0.7,0.2


In [9]:
SBVGPAdaptedResult()

sbvp_id,sbvp_trainer_id,adapt_prior_seed seed for adapt prior model,prior_fp_id to index into FlowPriorConfig,prior_trainer_id to index into FPTrainerConfig,likelihood_id to index into LikelihoodConfig,likelihood_trainer_id to index into LLTrainerConfig,orig_dl_id to index into DataLoaderConfig used for the prior and likelihood training,adapt_prior_trainer_id to index into AdaptPriorTrainer,alt_dl_id to index into AltDataLoaderConfig,n_samples number of samples to generate,data_seed seed for generating samples,"train_ll_mean mean per dimension, per sample, in nats",train_ll_sem standard error of the mean,val_ll_mean,val_ll_sem,test_ll_mean,test_ll_sem,"train_ll_mean_sample mean per dimension, per sample, in nats",train_ll_sem_sample standard error of the mean,val_ll_mean_sample,val_ll_sem_sample,test_ll_mean_sample,test_ll_sem_sample,tracker_output,eval_output,model


In [7]:
DataLoaderConfig()

id,data_fname,train_prop,val_prop
05977a317062b759857ee411a2e60648,/src/project/data/synthetic/haefner_2afc/haefner_2neuron_task1.pkl,0.7,0.2
260a5ea8175f75eaef132f42873ad14a,/src/project/data/synthetic/haefner_2afc/original_haefner_2afc_task_1_dataset.pkl,0.7,0.2
4477b5e82704db0bc19727864c7ef5aa,/src/project/data/synthetic/haefner_2afc/haefner_4neuron_task1.pkl,0.7,0.2
94efb58694007205fac996d7963f88c5,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_highdelta_task2_dataset.pkl,0.7,0.2
b8379e7d6998fc94a08a9a3742eec12d,/src/project/data/synthetic/haefner_2afc/flat_haefner_dataset.pkl,0.7,0.2
d74090584b0b974c4444a5ec64c3d87d,/src/project/data/synthetic/haefner_2afc/haefner_model_2neuron_highdelta_task2_dataset.pkl,0.7,0.2
f1ae78885d2ace1ba976199d4cf1a4d6,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_task1_dataset.pkl,0.7,0.2
f7b32dd97feda9f34e2b47e24fa3d18b,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_highdelta_task1_dataset.pkl,0.7,0.2


In [3]:
SBVGPAdaptedResult()

NameError: name 'SBVGPAdaptedResult' is not defined

In [6]:
SBVGPTrainerConfig()

id,lr,weight_decay,n_epochs,batch_size,early_stopping_threshold,early_stopping_patience
a2a0e79d4c785fead2a84d56420b6063,0.001,0.001,500,128,10,10
f89651063b51487dcdf4041336ef89db,0.001,0.001,250,128,10,10


In [2]:
DataLoaderConfig()

id,data_fname,train_prop,val_prop
05977a317062b759857ee411a2e60648,/src/project/data/synthetic/haefner_2afc/haefner_2neuron_task1.pkl,0.7,0.2
260a5ea8175f75eaef132f42873ad14a,/src/project/data/synthetic/haefner_2afc/original_haefner_2afc_task_1_dataset.pkl,0.7,0.2
4477b5e82704db0bc19727864c7ef5aa,/src/project/data/synthetic/haefner_2afc/haefner_4neuron_task1.pkl,0.7,0.2
94efb58694007205fac996d7963f88c5,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_highdelta_task2_dataset.pkl,0.7,0.2
b8379e7d6998fc94a08a9a3742eec12d,/src/project/data/synthetic/haefner_2afc/flat_haefner_dataset.pkl,0.7,0.2
d74090584b0b974c4444a5ec64c3d87d,/src/project/data/synthetic/haefner_2afc/haefner_model_2neuron_highdelta_task2_dataset.pkl,0.7,0.2
f1ae78885d2ace1ba976199d4cf1a4d6,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_task1_dataset.pkl,0.7,0.2
f7b32dd97feda9f34e2b47e24fa3d18b,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_highdelta_task1_dataset.pkl,0.7,0.2


In [5]:
AdaptPriorResult & "dl_id = '94efb58694007205fac996d7963f88c5'"

seed,prior_fp_id to index into FlowPriorConfig,prior_trainer_id to index into FPTrainerConfig,likelihood_id to index into LikelihoodConfig,likelihood_trainer_id to index into LLTrainerConfig,orig_dl_id to index into DataLoaderConfig used for the prior and likelihood training,trainer_id,dl_id,"train_marginal_obs_ll_mean mean per trial, per sample, in nats",train_marginal_obs_ll_sem standard error of the mean,val_marginal_obs_ll_mean,val_marginal_obs_ll_sem,test_marginal_obs_ll_mean,test_marginal_obs_ll_sem,"train_prior_ll_mean mean per trial, per sample, in nats",train_prior_ll_sem standard error of the mean,val_prior_ll_mean,val_prior_ll_sem,test_prior_ll_mean,test_prior_ll_sem,tracker_output,eval_output,model trained joint model NOT just the prior
42,89c1053a65023b042dc63f7f852bb5b0,c40a50ce9c77369770dddd5129836477,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,f7b32dd97feda9f34e2b47e24fa3d18b,132c5a41de356eda4032103ef56e8126,94efb58694007205fac996d7963f88c5,119.97028350830078,1.954919457435608,119.29264068603516,4.956930160522461,119.5291519165039,4.722094535827637,-2.870461940765381,0.13145679235458374,-2.8621463775634766,0.26104775071144104,-2.8385438919067383,0.37460264563560486,=BLOB=,=BLOB=,=BLOB=
100,89c1053a65023b042dc63f7f852bb5b0,c40a50ce9c77369770dddd5129836477,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,f7b32dd97feda9f34e2b47e24fa3d18b,38da520d4873f6c53b3dcf33746e62ab,94efb58694007205fac996d7963f88c5,120.12609100341797,1.7330855131149292,119.39483642578125,4.706287860870361,119.5165786743164,5.07781982421875,-2.7707297801971436,0.13531842827796936,-2.764779806137085,0.2669220268726349,-2.739873170852661,0.3822442591190338,=BLOB=,=BLOB=,=BLOB=
100,89c1053a65023b042dc63f7f852bb5b0,c40a50ce9c77369770dddd5129836477,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,f7b32dd97feda9f34e2b47e24fa3d18b,eabf636932f56d44dcddf7300cf67f63,94efb58694007205fac996d7963f88c5,119.54377746582031,6.087575435638428,115.10975646972656,34.28639221191406,119.61746215820312,5.428261756896973,-2.532292604446411,0.1807646006345749,-2.5340750217437744,0.3516136705875397,-2.5060253143310547,0.49754953384399414,=BLOB=,=BLOB=,=BLOB=


In [4]:
DataLoaderConfig()

id,data_fname,train_prop,val_prop
05977a317062b759857ee411a2e60648,/src/project/data/synthetic/haefner_2afc/haefner_2neuron_task1.pkl,0.7,0.2
260a5ea8175f75eaef132f42873ad14a,/src/project/data/synthetic/haefner_2afc/original_haefner_2afc_task_1_dataset.pkl,0.7,0.2
4477b5e82704db0bc19727864c7ef5aa,/src/project/data/synthetic/haefner_2afc/haefner_4neuron_task1.pkl,0.7,0.2
94efb58694007205fac996d7963f88c5,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_highdelta_task2_dataset.pkl,0.7,0.2
b8379e7d6998fc94a08a9a3742eec12d,/src/project/data/synthetic/haefner_2afc/flat_haefner_dataset.pkl,0.7,0.2
d74090584b0b974c4444a5ec64c3d87d,/src/project/data/synthetic/haefner_2afc/haefner_model_2neuron_highdelta_task2_dataset.pkl,0.7,0.2
f1ae78885d2ace1ba976199d4cf1a4d6,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_task1_dataset.pkl,0.7,0.2
f7b32dd97feda9f34e2b47e24fa3d18b,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_highdelta_task1_dataset.pkl,0.7,0.2


In [ ]:
SIResult & "dl_id = 'f7b32dd97feda9f34e2b47e24fa3d18b'"

In [ ]:
SIResult & "dl_id = '94efb58694007205fac996d7963f88c5'"